# CSI5341 DEIM: Colab runner
Workflow: edit locally, `git push`, then re-run the **Train** cell here (it pulls the latest commit first).
First time in a session: run cells 1 and 2. Runtime type must be **GPU (T4)**.

In [ ]:
# 1. Settings + Drive
BRANCH = "CSI5341-Project"
DRIVE_ROOT = "/content/drive/MyDrive/CSI5341-DEIM"

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2. Setup: clone/pull code, install deps, restore data and weights (re-runnable, fast after the first time)
!mkdir -p /content && cd /content && ( [ -d CSI5341-DEIM ] || git clone -q -b {BRANCH} https://github.com/AliAdenAwil/CSI5341-DEIM.git )
!bash /content/CSI5341-DEIM/tools/colab_setup.sh {BRANCH} {DRIVE_ROOT}
%cd /content/CSI5341-DEIM

In [ ]:
# 3. Train: pulls the latest pushed code, then trains. Resumes automatically if the run already has last.pth on Drive.
CONFIG   = "configs/deim_dfine/deim_hgnetv2_n_pets.yml"
RUN_NAME = "pilot_mal_dense_s0"      # one folder per (arm, seed) under DRIVE_ROOT/outputs
SEED     = 0
FROM_COCO = True                     # False for the ImageNet-backbone-only reference run
EXTRA    = ""                        # extra -u overrides, e.g. "epoches=10 train_dataloader.total_batch_size=16"

import os, shlex
!bash tools/colab_setup.sh {BRANCH} {DRIVE_ROOT} > /dev/null && git log -1 --format='code: %h %s'

out = f"{DRIVE_ROOT}/outputs/{RUN_NAME}"
last = f"{out}/last.pth"
if os.path.exists(last):
    init = f"-r {shlex.quote(last)}"
    print("resuming from", last)
elif FROM_COCO:
    init = "-t weights/deim_dfine_hgnetv2_n_coco_160e.pth"
else:
    init = ""
overrides = f"HGNetv2.local_model_dir={DRIVE_ROOT}/weights/hgnetv2/ DEIMCriterion.log_match_iou=True {EXTRA}"
!python train.py -c {CONFIG} {init} --seed {SEED} --use-amp --output-dir {shlex.quote(out)} -u {overrides}

In [ ]:
# 4. Quick look at a run
import pandas as pd, json
print(pd.read_csv(f"{out}/match_iou.csv").tail())
ap = [json.loads(l)["test_coco_eval_bbox"][0] for l in open(f"{out}/log.txt")]
print("val AP per epoch:", [round(a, 3) for a in ap])